# Multi-class Segmentation과 U-Net 구현하기

## 0. 환경설정(라이브러리 임포트)

In [ ]:
import torch, os, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from glob import glob
from tqdm.notebook import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import models, transforms

In [ ]:
!unzip car.zip

## 1. 데이터 확인

In [ ]:
image_list = glob('./car-segmentation/images/*')
mask_list = glob('./car-segmentation/masks/*')
len(image_list), len(mask_list) # (211, 211)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.ravel()
for i in range(4):
  img_path = image_list[i]
  mask_path = image_list[i].replace('images', 'masks')
  axes[2 * i].imshow(Image.open(img_path))
  axes[2 * i].set_title(f'Image {i + 1}')
  axes[2 * i].axis('off')
  axes[2 * i + 1].imshow(Image.open(mask_path))
  axes[2 * i + 1].set_title(f'Mask {i + 1}')
  axes[2 * i + 1].axis('off')

## 2. 클래스 확인

In [ ]:
with open('./car-segmentation/classes.txt', 'r') as f:
  cls_list = f.read().split(',')
  cls_list = [cls.strip() for cls in cls_list]
num_classes = len(cls_list)
print(num_classes , '\n', cls_list)

## 2-1. 마스크 고유값 확인

In [ ]:
unique, counts = np.unique(mask_list, return_counts=True)
print(np.asarray((unique, counts)).T)

In [ ]:
for mask_path in mask_list:
  mask = Image.open(mask_path)
  mask = np.array(mask)
  print(mask)
  print(np.unique(mask))
  break

## 2-2. 이미지-마스크 겹쳐 그리기

`torchvision.utils.draw_segmentation_masks` 활용

In [ ]:
def imgs_with_masks(img_path, start):
  img_dir = img_path[start: start + 4 ]
  fig, axes = plt.subplots(1, 4, figsize=(15, 8))
  for i, img_path in enumerate(img_dir):
    # 이미지 및 마스크 불러오기
    img = torchvision.io.read_image(img_path)
    # 마스크에서 불필요한 채널 제거
    mask = torchvision.io.read_image(img_path.replace('images', 'masks')
    ).squeeze(0)
    # 마스크를 원-핫 인코딩하여 5개의 클래스로 변환
    one_hot_mask = F.one_hot(mask.to(torch.int64),
                             num_classes=num_classes).permute(2, 0, 1).float()
    print(img.shape, mask.shape, one_hot_mask.shape, one_hot_mask.dtype)
    # 각 마스크 채널을 True/False로 변환하여 오버레이 생성
    overlayed = torchvision.utils.draw_segmentation_masks(img,
        one_hot_mask.to(torch.bool), alpha=0.5,
        colors=['black', 'green', 'blue', 'yellow', 'purple'])
    axes[i].imshow(overlayed.permute(1, 2, 0)) # plot에 맞게 채널을 맨 뒤로
    axes[i].set_title(f'Image {i + 1}')
    axes[i].axis('off')

In [ ]:
imgs_with_masks(image_list, 5)

## 2-3. 3차원이 아닌 이미지 제거

In [ ]:
exclude = []
for img_path in image_list:
  img = Image.open(img_path)
  img = np.array(img)
  if img.ndim != 3:
    exclude.append(img_path)
    print(img.shape, img_path, img.ndim)
print(len(exclude))

In [ ]:
for ex in exclude:
  image_list.remove(ex)
  mask_list.remove(ex.replace('images', 'masks'))
len(image_list)

## 3. 나만의 Dataset 클래스 작성

- Image: (B, 3, H, W)
- Mask: (B, H, W)

In [ ]:
class CarDataset(Dataset):
  def __init__(self, image_list, mask_list, transform_img=None,
               transform_mask=None, num_classes=num_classes):
    self.image_list = image_list
    self.mask_list = mask_list
    self.transform_img = transform_img
    self.transform_mask = transform_mask
    self.num_classes = num_classes
  def __len__(self):
    return len(self.image_list)
  def __getitem__(self, idx):
    image_path = self.image_list[idx]
    mask_path = self.mask_list[idx]
    image = Image.open(image_path)
    mask = torchvision.io.read_image(mask_path)
    # 이미지 Channel : 4 --> RGBA
    if image.mode == 'RGBA' :
      image = image.convert('RGB')
    image = self.transform_img(image)
    mask = self.transform_mask(mask).squeeze(0).to(torch.long)
    return image, mask

In [ ]:
transform_img = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
]) # X 이미지 전처리
transform_mask = transforms.Compose([
    transforms.Resize((224, 224))
]) # y mask 전처리

In [ ]:
car_dataset = CarDataset(image_list, mask_list, transform_img, transform_mask)
len(car_dataset)

In [ ]:
# train / val을 8:2로 분리
train_size = int(len(car_dataset) * 0.8)
val_size = len(car_dataset) - train_size
train_ds, val_ds = random_split(car_dataset, [train_size, val_size])
print(len(train_ds), len(val_ds))
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)
# 데이터 로더로부터 첫 번째 배치만 받아서 열기
imgs, mask = next(iter(train_dl))
imgs.shape, mask.shape

## 4. 학습 및 추론 준비
### 4-1. FCN8s 모델 작성

In [ ]:
class FCN8s(nn.Module):
  def __init__(self, n_class=num_classes):
    super(FCN8s, self).__init__()
    vgg = models.vgg16(pretrained=True)
    features = list(vgg.features.children())
    # VGG16의 각 블록을 PyTorch Sequential로 구성
    self.features3 = nn.Sequential(*features[0:17])
    self.features4 = nn.Sequential(*features[17:24])
    self.features5 = nn.Sequential(*features[24:])
    # 추가 Conv 레이어
    self.conv6 = nn.Conv2d(512, 4096, kernel_size=7, padding=3)
    self.conv7 = nn.Conv2d(4096, 4096, kernel_size=1)
    # FCN에서 사용할 1x1 Conv (Pointwise Conv)
    self.conv1x1_pool3 = nn.Conv2d(256, n_class, kernel_size=1)
    self.conv1x1_pool4 = nn.Conv2d(512, n_class, kernel_size=1)
    self.conv1x1_pool5 = nn.Conv2d(4096, n_class, kernel_size=1)
    # Transposed Convolutions for upsampling
    self.upconv2 = nn.ConvTranspose2d(n_class, n_class, kernel_size=4, stride=2, padding=1) # 입력 영상을 2배 확대
    self.upconv8 = nn.ConvTranspose2d(n_class, n_class, kernel_size=8, stride=8, padding=0) # 입력 영상을 8배 확대
  def forward(self, x):
    # VGG16 인코더 부분
    p3 = self.features3(x)    # pool 3 (256, H/8, W/8)
    p4 = self.features4(p3)    # pool 4 (512, H/16, W/16)
    p5 = self.features5(p4)    # pool 5 (512, H/32, W/32)
    p5 = F.relu(self.conv6(p5))    # conv 6 (4096, H/32, W/32)
    p5 = F.relu(self.conv7(p5))    # conv 7 (4096, H/32, W/32)
    # 디코더 부분(FCN8s Head 부분)
    score = self.conv1x1_pool5(p5)
    score = self.upconv2(score) + self.conv1x1_pool4(p4)
    score = self.upconv2(score) + self.conv1x1_pool3(p3)
    output = self.upconv8(score)
    return output

In [ ]:
# 모델 인스턴스 생성
model = FCN8s().cuda()
input_images = torch.randn(1, 3, 224, 224).cuda()
output = model(input_images)
# (batch, n_class, height, weight)
output.shape

#### Transposed Convolution의 핵심 원리

- 일반 Convolution의 역연산처럼 동작하여 공간 해상도를 키우는 역할을 합니다.
- 출력 크기 공식은 다음과 같습니다.
$$H_o = (H_i - 1) * stride - 2 * padding + kernel\_size + output\_padding$$
- 여기서 $H_i$는 입력 크기, $H_o$는 출력크기입니다.

` self.upconv2 = nn.ConvTranspose2d(n_class, n_class, kernel_size=4, stride=2, padding=1)` 입력 크기 $H_i$를 넣으면 출력 크기 공식 $(H_i - 1) * 2 - 2 + 4$ 는 2배

` self.upconv2 = nn.ConvTranspose2d(n_class, n_class, kernel_size=8, stride=8, padding=0)` 입력 크기 $H_i$를 넣으면 출력 크기 공식 $(H_i - 1) * 8 + 8$ 는 8배

### 4-2. 학습 루프 설정

In [ ]:
criterion = nn.CrossEntropyLoss()
optim = torch.optim.Adam(params = model.parameters(), lr=5e-4)
epochs = 15

#### 평가지표 (Multi Class Segmentation이므로 mean~ 활용)

- mean IoU
- mean Pixel Accuracy


[TODO] 두 지표를 다중 클래스 세그멘테이션 모델에 적용하도록 바꿔봅니다.

In [ ]:
def IoU(output, mask, num_classes=num_classes):
  """
  1. 각 클래스별로 IoU 계산
  2. 클래스별 IoU를 평균하여 전체 IoU 계산
  """
  output = torch.argmax(output, dim=1) # 각 픽셀에서 가장 높은 값(클래스) 선택
  iou_per_class = []
  for cls in range(num_classes):
    output_cls = (output == cls).float() # 현재 클래스에 대한 예측값
    mask_cls = (mask == cls).float()     # 현재 클래스에 대한 실제값
    intersection = (output_cls * mask_cls).sum()
    union = output_cls.sum() + mask_cls.sum() - intersection
    if union == 0:
      iou_per_class.append(torch.tensor(1.0)) # 둘 다 배경(0)이면 IoU는 1로 처리
    else:
      iou_per_class.append((intersection / union)) # .item()이 없어야 torch.tensor형태(torch.stack 또는 torch.mean에 입력으로 활용 가능)
  return torch.mean(torch.stack(iou_per_class))
def PA(output, mask):
  """
  1. 각 클래스별로 IoU 계산
  2. 클래스별 IoU를 평균하여 전체 IoU 계산
  """
  output = torch.argmax(output, dim=1) # 각 픽셀에서 가장 높은 값(클래스) 선택
  correct = torch.sum(output == mask)
  total = torch.numel(mask)
  return correct / total

### 4-3. 학습 루프

In [ ]:
def train_and_validate(model, train_loader, val_loader, optim, criterion, epochs, num_classes):
  train_losses = []
  train_IoUs = []
  train_PAs = []
  val_losses = []
  val_IoUs = []
  val_PAs = []
  for epoch in range(epochs):
    train_loss, train_IoU, train_PA = 0, 0, 0
    model.train()
    for images, masks in tqdm(train_loader):
      images = images.cuda()
      masks = masks.cuda()
      optim.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, masks)
      loss.backward()
      optim.step()
      train_loss += loss.item()
      train_IoU += IoU(outputs.detach().cpu(), masks.detach().cpu(), num_classes).item()
      train_PA += PA(outputs.detach().cpu(), masks.detach().cpu()).item()
    train_loss /= len(train_loader)
    print(f"Epoch: {epoch+1}, Train Loss: {train_loss}, Train IoU: {train_IoU / len(train_loader)}, Train PA: {train_PA / len(train_loader)}")
    train_losses.append(train_loss)
    train_IoUs.append(train_IoU / len(train_loader))
    train_PAs.append(train_PA / len(train_loader))
    val_loss, val_IoU, val_PA = 0, 0, 0
    model.eval() ## validation용 알고리즘 적용
    with torch.no_grad(): # gradient 계산이 안 되도록 scope 지정
      for images, masks in tqdm(val_loader):
        images = images.cuda()
        masks = masks.cuda()
        outputs = model(images)
        loss = criterion(outputs, masks)
        val_loss += loss.item()
        val_IoU += IoU(outputs.detach().cpu(), masks.detach().cpu(), num_classes).item()
        val_PA += PA(outputs.detach().cpu(), masks.detach().cpu()).item()
      val_loss /= len(val_loader)
      print(f"Epoch: {epoch+1}, Val Loss: {val_loss}, Val IoU: {val_IoU / len(val_loader)}, Val PA: {val_PA / len(val_loader)}")
      val_losses.append(val_loss)
      val_IoUs.append(val_IoU / len(val_loader))
      val_PAs.append(val_PA / len(val_loader))
  return train_losses, train_IoUs, train_PAs, val_losses, val_IoUs, val_PAs

In [ ]:
train_losses, train_IoUs, train_PAs, val_losses, val_IoUs, val_PAs = train_and_validate(model, train_dl, val_dl, optim, criterion, epochs, num_classes)

## 5. 시각화
### 5-1. 로그(loss, mIoU, PAs) 시각화

In [ ]:
plt.plot(train_losses, label='train_loss')
plt.plot(val_losses, label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()
plt.plot(train_IoUs, label='train_mIoU')
plt.plot(val_IoUs, label='val_mIoU')
plt.xlabel('Epoch')
plt.ylabel('Mean Intersection Over Union')
plt.legend()
plt.show()
plt.plot(train_PAs, label='train_PAs')
plt.plot(val_PAs, label='val_PAs')
plt.xlabel('Epoch')
plt.ylabel('Pixel Accuracy')
plt.legend()
plt.show()

### 5-2. Image, Mask, Pred 시각화

In [ ]:
def plot_batch(model, data_loader):
  model.eval()
  with torch.no_grad():
    for img, mask in tqdm(data_loader):
      img, mask = img.cuda(), mask.cuda()
      output = model(img)
      # 예측값(ouput)에서 가장 높은 클래스 인덱스 선택 (배치에서 각 픽셀별로 argmax연산)
      output = torch.argmax(output, dim=1).cpu().numpy() # (B, H, W)
      img = img.cpu().numpy().transpose(0, 2, 3, 1) # (B, H, W, 3)
      mask = mask.cpu().numpy() # (B, H, W)
      break
  for i in range(data_loader.batch_size):
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    # 원본 이미지 시각화
    ax[0].imshow(img[i])
    ax[0].set_title('Original Image')
    # 마스크 시각화 (vmin=0, vmax=5로 클래스 범위를 설정)
    ax[1].imshow(mask[i], cmap='gray', vmin=0, vmax=5)
    ax[1].set_title('Ground Truth Mask')
    # 예측값 시각화 (vmin=0, vmax=5로 클래스 범위를 설정)
    ax[2].imshow(output[i], cmap='gray', vmin=0, vmax=5)
    ax[2].set_title('Predicted Mask')
    plt.show()
plot_batch(model, val_dl)